# Generator Consistency Report (GMM vs BGMM)

Runs multi-seed dual-generator scoring on wrought data and reports how consistently each generator wins.

Outputs:
- `generator_consistency_runs.csv` (one row per seed)
- `generator_consistency_summary.csv` (aggregated wins and score statistics)

Environment variables (optional):
- `GENERATOR_REPORT_SEEDS` (default `10`)
- `GENERATOR_REPORT_SAMPLES` (default `5000`)
- `GENERATOR_REPORT_RUNS_PATH` (default `generator_consistency_runs.csv`)
- `GENERATOR_REPORT_SUMMARY_PATH` (default `generator_consistency_summary.csv`)

In [1]:
import os
import numpy as np
import pandas as pd

try:
    from utils import (
        load_hyperparams,
        load_backward_gmm_params,
        load_backward_generator_params,
        load_backward_selection_config,
        get_default_hyperparams,
    )
    from synthetic_generation_core import load_wrought, run_dual_generator_once, env_int
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import (
        load_hyperparams,
        load_backward_gmm_params,
        load_backward_generator_params,
        load_backward_selection_config,
        get_default_hyperparams,
    )
    from synthetic_generation_core import load_wrought, run_dual_generator_once, env_int

WROUGHT_PATH = 'wrought_alloys_final.csv'
N_SEEDS = env_int('GENERATOR_REPORT_SEEDS', 10)
N_SAMPLES = env_int('GENERATOR_REPORT_SAMPLES', 5000)
RUNS_PATH = os.getenv('GENERATOR_REPORT_RUNS_PATH', 'generator_consistency_runs.csv')
SUMMARY_PATH = os.getenv('GENERATOR_REPORT_SUMMARY_PATH', 'generator_consistency_summary.csv')
SEED_START = env_int('GENERATOR_REPORT_SEED_START', 42)
SEED_STEP = env_int('GENERATOR_REPORT_SEED_STEP', 10)

SEEDS = [SEED_START + i * SEED_STEP for i in range(N_SEEDS)]
print(f'Configured N_SEEDS={N_SEEDS}, N_SAMPLES={N_SAMPLES}, SEEDS={SEEDS}')

Configured N_SEEDS=5, N_SAMPLES=2000, SEEDS=[42, 52, 62, 72, 82]


In [2]:
df_real, target_list = load_wrought(WROUGHT_PATH)
print(f'Loaded wrought data: {df_real.shape}, targets={len(target_list)}')

wrought_config = load_hyperparams('wrought')
by_target = (wrought_config or {}).get('by_target') or {}
if not by_target:
    raise ValueError('wrought.by_target not in config. Run 01_hyperparameter_tuning_forward.ipynb first.')

gmm_params = load_backward_gmm_params('wrought') or get_default_hyperparams('GMM')
bgmm_params = load_backward_generator_params('wrought', 'BGMM') or get_default_hyperparams('BGMM')
selection_cfg = load_backward_selection_config('wrought') or get_default_hyperparams('BACKWARD_SYNTHETIC_SELECTION')

print('Loaded generator/selection configuration.')

Loaded wrought data: (868, 31), targets=13
Loaded generator/selection configuration.


In [3]:
run_rows = []
all_scores_long = []

for seed in SEEDS:
    result = run_dual_generator_once(
        df_real=df_real,
        by_target=by_target,
        gmm_params=gmm_params,
        bgmm_params=bgmm_params,
        selection_cfg=selection_cfg,
        get_default_hyperparams=get_default_hyperparams,
        n_samples=N_SAMPLES,
        random_seed=seed,
    )
    score_table = result['scores'].copy().reset_index(drop=True)
    all_scores_long.append(score_table.assign(seed=seed))

    row = {'seed': seed, 'winner': result['best_generator']}
    for generator_name in ['GMM', 'BGMM']:
        g_row = score_table[score_table['generator'] == generator_name]
        if g_row.empty:
            continue
        g = g_row.iloc[0]
        row[f'{generator_name.lower()}_backward_raw'] = float(g['backward_raw'])
        row[f'{generator_name.lower()}_forward_realism_raw'] = float(g['forward_realism_raw'])
        row[f'{generator_name.lower()}_composition_realism_raw'] = float(g['composition_realism_raw'])
        row[f'{generator_name.lower()}_backward_score'] = float(g['backward_score'])
        row[f'{generator_name.lower()}_forward_realism_score'] = float(g['forward_realism_score'])
        row[f'{generator_name.lower()}_composition_realism_score'] = float(g['composition_realism_score'])
        row[f'{generator_name.lower()}_total_score'] = float(g['total_score'])

    row['score_gap'] = row.get('gmm_total_score', np.nan) - row.get('bgmm_total_score', np.nan)
    run_rows.append(row)

runs_df = pd.DataFrame(run_rows)
runs_df = runs_df.sort_values('seed').reset_index(drop=True)
long_scores_df = pd.concat(all_scores_long, ignore_index=True) if all_scores_long else pd.DataFrame()

runs_df.to_csv(RUNS_PATH, index=False)
print(f'Saved run-level report: {RUNS_PATH}')
print(runs_df[['seed', 'winner', 'gmm_total_score', 'bgmm_total_score', 'score_gap']])

C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


Saved run-level report: generator_consistency_runs.csv
   seed winner  gmm_total_score  bgmm_total_score  score_gap
0    42    GMM              1.0               0.0        1.0
1    52    GMM              1.0               0.0        1.0
2    62    GMM              1.0               0.0        1.0
3    72    GMM              1.0               0.0        1.0
4    82    GMM              1.0               0.0        1.0


In [4]:
if runs_df.empty:
    raise RuntimeError('No consistency runs were produced.')

winner_counts = runs_df['winner'].value_counts().to_dict()

summary_rows = [
    {
        'metric': 'num_seeds',
        'value': int(len(runs_df)),
    },
    {
        'metric': 'gmm_wins',
        'value': int(winner_counts.get('GMM', 0)),
    },
    {
        'metric': 'bgmm_wins',
        'value': int(winner_counts.get('BGMM', 0)),
    },
    {
        'metric': 'gmm_win_rate',
        'value': float(winner_counts.get('GMM', 0) / len(runs_df)),
    },
    {
        'metric': 'bgmm_win_rate',
        'value': float(winner_counts.get('BGMM', 0) / len(runs_df)),
    },
    {
        'metric': 'mean_score_gap_gmm_minus_bgmm',
        'value': float(runs_df['score_gap'].mean()),
    },
    {
        'metric': 'std_score_gap_gmm_minus_bgmm',
        'value': float(runs_df['score_gap'].std(ddof=0)),
    },
]

for col in [c for c in runs_df.columns if c.endswith('_total_score') or c.endswith('_raw')]:
    summary_rows.append({'metric': f'mean_{col}', 'value': float(runs_df[col].mean())})
    summary_rows.append({'metric': f'std_{col}', 'value': float(runs_df[col].std(ddof=0))})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f'Saved summary report: {SUMMARY_PATH}')
print(summary_df)

Saved summary report: generator_consistency_summary.csv
                               metric     value
0                           num_seeds  5.000000
1                            gmm_wins  5.000000
2                           bgmm_wins  0.000000
3                        gmm_win_rate  1.000000
4                       bgmm_win_rate  0.000000
5       mean_score_gap_gmm_minus_bgmm  1.000000
6        std_score_gap_gmm_minus_bgmm  0.000000
7               mean_gmm_backward_raw  0.931690
8                std_gmm_backward_raw  0.013941
9        mean_gmm_forward_realism_raw  0.144595
10        std_gmm_forward_realism_raw  0.003916
11   mean_gmm_composition_realism_raw  0.219769
12    std_gmm_composition_realism_raw  0.028389
13               mean_gmm_total_score  1.000000
14                std_gmm_total_score  0.000000
15             mean_bgmm_backward_raw  0.954045
16              std_bgmm_backward_raw  0.006039
17      mean_bgmm_forward_realism_raw  0.217629
18       std_bgmm_forward_realis